# 1 · Basic nodes

Register plain Python functions as **nodes** with widget annotations. One `Annotated[T, Widget]` per parameter is the single source of truth for backend validation, execution, and frontend rendering.

In this notebook:
- Single-input / single-output nodes
- Multi-input nodes with dropdowns
- Sliders, checkboxes, textareas
- Multi-output (tuple return)
- Optional parameters (defaults)

In [1]:
from typing import Annotated

from conductor import NodeRegistry
from conductor.widgets import Checkbox, Dropdown, Output, Range, Text, Textarea

registry = NodeRegistry()

## Simple single-input / single-output

In [2]:
@registry.node("greet", version=1, name="Greeting", description="Generates a greeting")
def greet(
    name: Annotated[str, Text(label="Name", description="Who to greet")],
) -> Annotated[str, Output(label="Greeting")]:
    return f"Hello, {name}!"


# Registered nodes are still plain Python functions — call them directly:
greet("World")

'Hello, World!'

## Multiple inputs with a dropdown

Use `Dropdown(choices=[...])` for a fixed set of options. Any parameter with a default value becomes optional.

In [3]:
@registry.node(
    "format-name",
    version=1,
    name="Format Name",
    description="Formats a full name",
)
def format_name(
    first: Annotated[str, Text(label="First name")],
    last: Annotated[str, Text(label="Last name")],
    style: Annotated[
        str,
        Dropdown(label="Style", choices=["First Last", "Last, First", "LAST First"]),
    ] = "First Last",
) -> Annotated[str, Output(label="Full name")]:
    match style:
        case "First Last":
            return f"{first} {last}"
        case "Last, First":
            return f"{last}, {first}"
        case "LAST First":
            return f"{last.upper()} {first}"
        case _:
            return f"{first} {last}"


format_name("Ada", "Lovelace", style="LAST First")

'LOVELACE Ada'

## Range slider, checkbox, textarea

In [4]:
@registry.node(
    "truncate",
    version=1,
    name="Truncate",
    description="Truncates text to a max length",
)
def truncate(
    text: Annotated[str, Textarea(label="Text", rows=3)],
    max_length: Annotated[int, Range(label="Max length", min_val=1, max_val=500, step=1)] = 100,
    add_ellipsis: Annotated[bool, Checkbox(label="Add ...")] = True,
) -> Annotated[str, Output(label="Truncated")]:
    if len(text) <= max_length:
        return text
    truncated = text[:max_length]
    return f"{truncated}..." if add_ellipsis else truncated


truncate("The quick brown fox jumps over the lazy dog", max_length=19)

'The quick brown fox...'

## Multi-output node

Returning a tuple of `Annotated[T, Output(...)]` creates a node with multiple named outputs. At runtime each output is addressable as `output_1`, `output_2`, ...

In [5]:
@registry.node(
    "split-text",
    version=1,
    name="Split Text",
    description="Splits text into first line and rest",
)
def split_text(
    text: Annotated[str, Textarea(label="Text", rows=4)],
) -> tuple[
    Annotated[str, Output(label="First line")],
    Annotated[str, Output(label="Remaining")],
]:
    lines = text.split("\n", 1)
    first = lines[0]
    rest = lines[1] if len(lines) > 1 else ""
    return first, rest


split_text("hello\nworld\nagain")

('hello', 'world\nagain')

## Inspect the registry

Every registered node exposes its introspected signature. This is what the frontend reads to render the node palette and input widgets.

In [6]:
print(f"Registered {len(registry.all())} nodes:\n")
for node in registry.all():
    inputs = ", ".join(f"{i.name}: {i.type_str}" for i in node.inputs)
    outputs = ", ".join(f"{o.name}: {o.type_str}" for o in node.outputs)
    print(f"  {node.id}: {node.name}")
    print(f"    inputs:  ({inputs})")
    print(f"    outputs: ({outputs})")
    print()

Registered 4 nodes:

  greet@1: Greeting
    inputs:  (name: str)
    outputs: (result: str)

  format-name@1: Format Name
    inputs:  (first: str, last: str, style: str)
    outputs: (result: str)

  truncate@1: Truncate
    inputs:  (text: str, max_length: int, add_ellipsis: bool)
    outputs: (result: str)

  split-text@1: Split Text
    inputs:  (text: str)
    outputs: (output_1: str, output_2: str)

